# The Triton programming model: blocks, masks and pointers

How Triton turns CUDA's thousands of threads into one program per block of data, what tl.program_id, tl.arange, masks and pointer arithmetic do, how to write vector add and a row-wise softmax, how to debug with the interpreter, and what Triton decides for you (coalescing, shared memory, warps) versus what you still decide.

This notebook holds the code from the lesson **[The Triton programming model: blocks, masks and pointers](https://inferenceacademy.com/learn/kern-triton/the-triton-programming-model)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## The big idea: programs over blocks

## Vector add in Triton

In [ ]:
import torch, triton, triton.language as tl

@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(axis=0)                 # which block am I?
    offs = pid * BLOCK + tl.arange(0, BLOCK)    # the BLOCK indices I handle
    mask = offs < n                             # the last block may run past the end
    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + y, mask=mask)

def add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    out = torch.empty_like(x)
    n = out.numel()
    grid = lambda meta: (triton.cdiv(n, meta["BLOCK"]),)
    add_kernel[grid](x, y, out, n, BLOCK=1024)
    return out

x = torch.randn(1 << 26, device="cuda"); y = torch.randn_like(x)
torch.testing.assert_close(add(x, y), x + y)
ms = triton.testing.do_bench(lambda: add(x, y))
print(f"{ms:.3f} ms, {3 * x.numel() * 4 / ms / 1e6:.0f} GB/s")

## Debugging with the interpreter

In [ ]:
import os
os.environ["TRITON_INTERPRET"] = "1"
import torch, triton, triton.language as tl

@triton.jit
def show(x_ptr, n, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    x = tl.load(x_ptr + offs, mask=offs < n, other=0.0)
    print("pid", tl.program_id(0), "offs", offs, "x", x)

x = torch.arange(10, dtype=torch.float32)        # CPU tensor
show[(3,)](x, 10, BLOCK=4)

## Blocks, masks and padding, visually

## Row-wise softmax

In [ ]:
@triton.jit
def softmax_kernel(x_ptr, y_ptr, stride_row, n_cols, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols
    x = tl.load(x_ptr + row * stride_row + cols, mask=mask, other=-float("inf"))
    x = x - tl.max(x, axis=0)                   # subtract the row max for stability
    num = tl.exp(x)
    y = num / tl.sum(num, axis=0)
    tl.store(y_ptr + row * stride_row + cols, y, mask=mask)

def softmax(x: torch.Tensor) -> torch.Tensor:
    M, N = x.shape
    y = torch.empty_like(x)
    BLOCK = triton.next_power_of_2(N)
    softmax_kernel[(M,)](x, y, x.stride(0), N, BLOCK=BLOCK, num_warps=4 if BLOCK <= 2048 else 8)
    return y

### What to expect

In [ ]:
x = torch.randn(4096, 4096, device="cuda")
torch.testing.assert_close(softmax(x), torch.softmax(x, dim=-1))
for name, f in [("triton", lambda: softmax(x)), ("torch", lambda: torch.softmax(x, -1))]:
    ms = triton.testing.do_bench(f)
    print(f"{name}: {ms:.3f} ms, {2 * x.numel() * 4 / ms / 1e6:.0f} GB/s")

## What Triton does for you, and what it doesn't

## Reading Triton code in engines

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-triton/the-triton-programming-model